# Fabric Table Preview with PySpark

command to run the jupyter server for this notebook: 

`runjupyter (in any folder)`

equivalent to running this in /go-api:

```docker compose run --rm -p 8888:8888 serve jupyter notebook --ip=0.0.0.0 --port=8888 --no-browser --allow-root --notebook-dir=/home/ifrc```


In [162]:
import os
from pathlib import Path
from urllib.request import urlretrieve

POSTGRES_JDBC_VERSION = '42.7.4'
POSTGRES_JDBC_JAR = Path(f'/tmp/postgresql-{POSTGRES_JDBC_VERSION}.jar')
POSTGRES_JDBC_URL = (
    'https://repo1.maven.org/maven2/org/postgresql/postgresql/'
    f'{POSTGRES_JDBC_VERSION}/postgresql-{POSTGRES_JDBC_VERSION}.jar'
)

if not POSTGRES_JDBC_JAR.exists():
    urlretrieve(POSTGRES_JDBC_URL, POSTGRES_JDBC_JAR)

# Must be set before Spark JVM starts in this kernel
os.environ['PYSPARK_SUBMIT_ARGS'] = f'--jars {POSTGRES_JDBC_JAR} pyspark-shell'

from pyspark.sql import SparkSession

In [163]:
# Stop the current Spark session so this cell can be rerun safely
active_spark = SparkSession.getActiveSession()
if active_spark is not None:
    active_spark.stop()

SPARK_MASTER = os.getenv("SPARK_MASTER", "local[*]")  # go to .env in go-api to configure, default to local 

spark = (
    SparkSession.builder
    .appName('postgres-stock-inventory')
    .master(SPARK_MASTER)
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

spark

In [164]:
# These are provided by docker compose env + .env when running notebook in `serve` container
PG_HOST = os.getenv('DJANGO_DB_HOST', 'db')
PG_PORT = os.getenv('DJANGO_DB_PORT', '5432')
PG_DB = os.environ['DJANGO_DB_NAME']
PG_USER = os.environ['DJANGO_DB_USER']
PG_PASSWORD = os.environ['DJANGO_DB_PASS']

jdbc_url = f'jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}'

# Read only table names in public schema to discover what you can load
table_list_query = """(
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name
) t"""

tables_df = (
    spark.read.format('jdbc')
    .option('url', jdbc_url)
    .option('dbtable', table_list_query)
    .option('user', PG_USER)
    .option('password', PG_PASSWORD)
    .option('driver', 'org.postgresql.Driver')
    .load()
)

print('Total public tables:', tables_df.count())
tables_df.show(5, truncate=False)

Total public tables: 351
+------------------------+
|table_name              |
+------------------------+
|api_action              |
|api_actionstaken        |
|api_actionstaken_actions|
|api_admin2              |
|api_admin2geoms         |
+------------------------+
only showing top 5 rows



In [165]:
# Example: load one simple Django table from go-api/api/models.py
table_name = 'api_CleanedFrameworkAgreement'

df_sample = (
    spark.read.format('jdbc')
    .option('url', jdbc_url)
    .option('dbtable', table_name)
    .option('user', PG_USER)
    .option('password', PG_PASSWORD)
    .option('driver', 'org.postgresql.Driver')
    .load()
)

print('Table:', table_name)
df_sample.show(10, truncate=False)

Table: api_CleanedFrameworkAgreement
+----+------------+--------------+-------------------------------------+--------------------------------------+---------------+---------+--------------+----------------------------+------------------------+--------------+------------------------+---------+------------------+------------------------------+--------------------------+--------------------------+-----+-------------------+-------------------+
|id  |agreement_id|classification|default_agreement_line_effective_date|default_agreement_line_expiration_date|workflow_status|status   |price_per_unit|pa_line_procurement_category|vendor_name             |vendor_country|region_countries_covered|item_type|item_category     |item_service_short_description|created_at                |updated_at                |owner|vendor_valid_from  |vendor_valid_to    |
+----+------------+--------------+-------------------------------------+--------------------------------------+---------------+---------+------------

In [166]:
# Load all required tables from Postgres as DataFrames
print("Loading tables from Postgres...")

dimwarehouse_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_dimwarehouse")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

dimproduct_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_dimproduct")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

diminventorytransactionline_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_diminventorytransactionline")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

diminventorytransaction_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_diminventorytransaction")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

diminventoryowner_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_diminventoryowner")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

dimproductcategory_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_dimproductcategory")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

diminventoryitem_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_diminventoryitem")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

diminventoryitemstatus_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_diminventoryitemstatus")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

dimproductreceiptline_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_dimproductreceiptline")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

print("✓ All tables loaded successfully")
print(f"  - dimwarehouse: {dimwarehouse_df.count()} rows")
print(f"  - dimproduct: {dimproduct_df.count()} rows")
print(f"  - diminventorytransactionline: {diminventorytransactionline_df.count()} rows")
print(f"  - diminventorytransaction: {diminventorytransaction_df.count()} rows")
print(f"  - diminventoryowner: {diminventoryowner_df.count()} rows")
print(f"  - dimproductcategory: {dimproductcategory_df.count()} rows")
print(f"  - diminventoryitem: {diminventoryitem_df.count()} rows")
print(f"  - diminventoryitemstatus: {diminventoryitemstatus_df.count()} rows")
print(f"  - dimproductreceiptline: {dimproductreceiptline_df.count()} rows")


Loading tables from Postgres...
✓ All tables loaded successfully
  - dimwarehouse: 1172 rows
  - dimproduct: 27640 rows
  - diminventorytransactionline: 24799 rows
  - diminventorytransaction: 21636 rows
  - diminventoryowner: 109 rows
  - dimproductcategory: 3273 rows
  - diminventoryitem: 27506 rows
  - diminventoryitemstatus: 6 rows
  - dimproductreceiptline: 15649 rows


In [167]:
# Register all DataFrames as SQL temp views
dimwarehouse_df.createOrReplaceTempView("dimwarehouse")
dimproduct_df.createOrReplaceTempView("dimproduct")
dimproductcategory_df.createOrReplaceTempView("dimproductcategory")
diminventorytransactionline_df.createOrReplaceTempView("diminventorytransactionline")
diminventorytransaction_df.createOrReplaceTempView("diminventorytransaction")
diminventoryowner_df.createOrReplaceTempView("diminventoryowner")
diminventoryitem_df.createOrReplaceTempView("diminventoryitem")
diminventoryitemstatus_df.createOrReplaceTempView("diminventoryitemstatus")
dimproductreceiptline_df.createOrReplaceTempView("dimproductreceiptline")

print("✓ All tables registered as SQL views")


✓ All tables registered as SQL views


In [168]:
#filter diminventorytransaction before joining

df = spark.sql("""
    select 
        *
    from diminventorytransaction
    where 
        reference_category NOT ILIKE '%Weighted average inventory closing%'
        and NOT excluded_from_inventory_value
    order by id
"""
)


df.createOrReplaceTempView("diminventorytransaction")

spark.sql("""
    select 
        *
    from diminventorytransaction
    where 
        reference_category ILIKE '%Weighted average inventory closing%'
""").show()


+---+------------------+----------------+-----------------------------+
| id|reference_category|reference_number|excluded_from_inventory_value|
+---+------------------+----------------+-----------------------------+
+---+------------------+----------------+-----------------------------+



In [169]:
#filter warehouses down to only these specific warehouse codes

df = spark.sql("""
    select
        *
    from dimwarehouse
    WHERE id IN (
    'AE1DUB002',
    'AR1BUE002',
    'AU1BRI003',
    'ES1LAS001',
    'GT1GUA001',
    'HN1COM002',
    'MY1SEL001',
    'PA1ARR001',
    'TR1ISTA02'
)
""")

df.show()

df.createOrReplaceTempView("dimwarehouse")

+---------+----+--------------------+--------------------+-------+
|       id|site|                name|      postal_address|country|
+---------+----+--------------------+--------------------+-------+
|AE1DUB002| AE1|IFRC Reg - Dubai ...|International Hum...|    ARE|
|AR1BUE002| PA1|IFRC Sub-Reg - Ar...|Aeropuerto Intern...|    ARG|
|AU1BRI003| MY1|IFRC Sub-Reg - Br...|113 Bancroft Road...|    AUS|
|ES1LAS001| AE1|IFRC Sub-Reg - La...|IFRC - CENTRO LOG...|    ESP|
|GT1GUA001| PA1|IFRC Sub-Reg - Gu...|FEDERACIÓN INTERN...|    GTM|
|HN1COM002| PA1|IFRC Sub-Reg - Ho...|Anillo Periferico...|    HND|
|MY1SEL001| MY1|IFRC Reg - Malays...|INTEGRATED LOGIST...|    MYS|
|PA1ARR001| PA1|IFRC Reg - Panama...|Regional Logistic...|    PAN|
|TR1ISTA02| AE1|Kizilay Logistics...|Akfirat, Kizilay,...|    TUR|
+---------+----+--------------------+--------------------+-------+



In [170]:
#filter diminventorytransactionline 
#transactions where the owner code is IFRC
#specific status
#status must be OK
#must not be returned

df = spark.sql("""
    select * 
    from diminventorytransactionline
    where 
        owner not ilike '%#ifrc%'
        and status IN (
            'Deducted',
            'Purchased',
            'Received',
            'Reserved physical',
            'Sold'
            )
        and item_status = 'OK'
        and NOT packing_slip_returned
""")

df.createOrReplaceTempView("diminventorytransactionline")
#more specifically the #... within id must not be #ifrc


In [171]:
spark.sql("""
    select * from diminventorytransactionline 
""").show(10)

+----------+-----------+----------------+------------+------------------+-------+--------------------+---------+--------------+---------------------+----------------+--------+-------------+--------------+-----------+-------------+------------+------------------+----------------------+---------+------------+---------------------+
|        id|item_status|item_status_name|     product|  voucher_physical|project|               batch|warehouse|         owner|inventory_transaction|project_category|activity|physical_date|financial_date|status_date|expected_date|    quantity|cost_amount_posted|cost_amount_adjustment|   status|packing_slip|packing_slip_returned|
+----------+-----------+----------------+------------+------------------+-------+--------------------+---------+--------------+---------------------+----------------+--------+-------------+--------------+-----------+-------------+------------+------------------+----------------------+---------+------------+---------------------+
|563714

In [178]:
spark.sql("""
    select * from dimproductcategory
""").show(truncate=False)

+-------------+-----------------------------------------------------+--------------------+-----+
|category_code|name                                                 |parent_category_code|level|
+-------------+-----------------------------------------------------+--------------------+-----+
|5637144576   |IFRC/ICRC Catalog                                    |NULL                |1    |
|5637144577   |IFRC Item and Services                               |5637144576          |2    |
|5637145743   |Sundry/Other General &                               |FBAF                |5    |
|5637151327   |IT Vendor Services                                   |S                   |4    |
|5637156576   |Coffee powder                                        |5637145743          |6    |
|5637164826   |NULL                                                 |NULL                |NULL |
|A            |Administration                                       |5637144577          |3    |
|AAUD         |Audio Accessori

In [183]:
spark.sql("""
    select
        w.id as warehouse_id
        ,w.name as warehouse
        ,p.name as item_name
        ,quantity
    from dimwarehouse w
    join diminventorytransactionline itl
        on w.id = itl.warehouse
    join diminventorytransaction it
        on itl.inventory_transaction = it.id
    join dimproduct p
        on itl.product = p.id
""").show(1000)

+------------+--------------------+--------------------+---------------+
|warehouse_id|           warehouse|           item_name|       quantity|
+------------+--------------------+--------------------+---------------+
|   MY1SEL001|IFRC Reg - Malays...|KITCHEN SET famil...|    4000.000000|
|   PA1ARR001|IFRC Reg - Panama...|JERRYCAN, collaps...|     700.000000|
|   PA1ARR001|IFRC Reg - Panama...|BUCKET, plastic, ...|   -5000.000000|
|   AE1DUB002|IFRC Reg - Dubai ...|BLANKET, SYNTHETI...|   10000.000000|
|   AE1DUB002|IFRC Reg - Dubai ...|KITCHEN SET famil...|   -3650.000000|
|   AE1DUB002|IFRC Reg - Dubai ...|SHELTER TOOL KIT,...|    -500.000000|
|   ES1LAS001|IFRC Sub-Reg - La...|TARPAULINS, woven...|    2000.000000|
|   PA1ARR001|IFRC Reg - Panama...|COVERALL size XL,...|   -5400.000000|
|   PA1ARR001|IFRC Reg - Panama...|TARPAULINS, woven...|    -800.000000|
|   PA1ARR001|IFRC Reg - Panama...|          SOLAR LAMP|    8000.000000|
|   AU1BRI003|IFRC Sub-Reg - Br...|KITCHEN SET fami